<div style="text-align: center; color:#000; font-family: Arial, sans-serif;
background: linear-gradient(135deg, #89c881 0%, #4caf50 100%);
padding: 20px; border-radius: 12px;
box-shadow: 0 4px 12px rgba(0,0,0,0.3); line-height: 1.6;">

<h2 style="margin:0 0 6px 0;">Series de Tiempo con Redes Neuronales</h2>
<h4 style="margin:0 0 4px 0;">Universidad Central</h4>
<h4 style="margin:0 0 4px 0;">Maestría en Analítica de Datos</h4>
<p style="margin:0 0 2px 0;"><strong>Docente:</strong> Wilson Sandoval Rodriguez</p>
<p style="margin:0;"><em>wsandovalr@ucentral.edu.co</em></p>
</div>

## Mapa del cuadernillo

Este cuadernillo cubre el flujo completo de modelado de series de tiempo
con redes neuronales aplicadas a datos de calidad del aire:

| # | Sección | Contenido |
|---|---------|-----------|
| 1 | Introducción | Motivación y contexto del Deep Learning en ST |
| 2 | Marco teórico | RNN, LSTM, GRU, tipos de problemas |
| 3 | Datos | Dataset de calidad del aire — Valencia, España |
| 4 | **EDA** | Valores faltantes, outliers, estacionalidad, correlaciones |
| 5 | División temporal | Train / Validación / Test sin solape |
| 6 | Escalado | MinMaxScaler sin fuga de información |
| 7 | Ventanas deslizantes | Construcción de secuencias para LSTM |
| 8 | LSTM univariado 1-paso | Modelo base con evaluación completa |
| 9 | LSTM multi-step | Horizonte de 24 a 120 horas |
| 10 | LSTM multivariado (N:1) | Todas las variables → predecir O₃ |
| **11** | **Comparativa de arquitecturas** | MLP, GRU, CNN-1D vs LSTM |
| 12 | Tabla comparativa | Todas las arquitecturas del estado del arte |
| 13 | Conclusiones | Interpretación y recomendaciones finales |

> **Variable objetivo:** Concentración de ozono (O₃) en µg/m³.
> **Frecuencia:** Horaria. **Período:** 2019-01-01 → 2023-12-31.

## 1. Introducción

![Series de Tiempo](https://github.com/Wilsonsr/Series-de-Tiempo/raw/main/Data/red_neuronal_1.png)

Las **redes neuronales artificiales** son modelos computacionales inspirados en el
funcionamiento del cerebro humano. Están formadas por nodos (neuronas artificiales)
organizados en capas, donde cada nodo recibe información, la procesa y la transmite
a los nodos siguientes. Esta arquitectura permite aprender patrones complejos
directamente desde los datos, incluso cuando las relaciones no son lineales.

En el contexto de las **series de tiempo ambientales**, las redes neuronales son
especialmente valiosas porque:

- La calidad del aire tiene **patrones cíclicos** (hora del día, estación del año).
- Existen **dependencias temporales** entre contaminantes (e.g., NO₂ precede al O₃).
- Las relaciones entre variables son **no lineales** y difíciles de capturar con
  modelos estadísticos clásicos como ARIMA.

Este cuadernillo usa datos reales de contaminación atmosférica de Valencia (España)
para ilustrar el ciclo completo de modelado, desde el análisis exploratorio hasta
la comparación de múltiples arquitecturas de redes neuronales.

## 2. Marco Teórico

### 2.1 Redes Neuronales Recurrentes (RNN)

Las **RNN** son redes diseñadas para datos **secuenciales**, donde el orden importa.
A diferencia de una red feedforward (MLP), la RNN mantiene un **estado oculto**
que se actualiza en cada paso temporal, permitiéndole "recordar" el pasado.

![redes_neuronales.png](https://github.com/Wilsonsr/Series-de-Tiempo/raw/main/Data/redes_neuronales.png)

**Limitación crítica:** Las RNN simples sufren el problema del
**gradiente desvaneciente** — durante el entrenamiento, los gradientes se hacen
tan pequeños que la red "olvida" información de pasos lejanos. Esto limita su
capacidad para capturar dependencias a largo plazo.

---

### 2.2 LSTM (Long Short-Term Memory)

Las **LSTM** fueron diseñadas específicamente para superar el problema del gradiente
desvaneciente. Su mecanismo central es la **celda de memoria** (cell state), que
puede mantener información relevante durante muchos pasos temporales.

![Diagrama LSTM](https://databasecamp.de/wp-content/uploads/lstm-architecture-1024x709.png)

Una celda LSTM tiene **tres puertas**:

| Puerta | Función | Activación |
|--------|---------|-----------|
| **Olvido** (Forget gate) | Decide qué parte del estado anterior eliminar | Sigmoide (0=olvidar, 1=mantener) |
| **Entrada** (Input gate) | Decide qué nueva información almacenar | Sigmoide + tanh |
| **Salida** (Output gate) | Decide qué parte del estado pasa como salida | Sigmoide + tanh |

```
h_t = output_gate ⊙ tanh(C_t)
C_t = forget_gate ⊙ C_{t-1} + input_gate ⊙ C̃_t
```

---

### 2.3 GRU (Gated Recurrent Unit)

La **GRU** es una versión simplificada de la LSTM con **dos puertas** en lugar de tres:

| Puerta | Función |
|--------|---------|
| **Actualización** (Update gate) | Controla cuánto del estado anterior conservar |
| **Reinicio** (Reset gate) | Controla qué parte del estado anterior usar para el nuevo candidato |

**Ventajas de GRU vs LSTM:**
- Menos parámetros → más rápida de entrenar.
- Rendimiento comparable en datasets de tamaño medio.
- Menos propensa a sobreajuste con pocos datos.

---

### 2.4 Tipos de problemas en series de tiempo

```
Entrada única → salida única (1:1)
  [x_t-24, ..., x_t] → [x_{t+1}]        Ejemplo: predecir O₃ de mañana

Múltiple → única (N:1)
  [x1_t, x2_t, ..., xN_t] → [y_{t+1}]  Ejemplo: PM2.5+NO2+CO → predecir O₃

Single-step:  predice 1 paso adelante
Multi-step:   predice H pasos adelante (horizonte H)
```

| Tipo | Entradas | Salida | Pasos futuros |
|------|----------|--------|---------------|
| 1:1 single-step | Una serie | 1 valor | 1 |
| 1:1 multi-step | Una serie | H valores | H |
| N:1 single-step | N series | 1 valor | 1 |
| N:1 multi-step | N series | H valores | H |
| N:M | N series | M series × H valores | H |

## 3. Instalación e importaciones

In [ ]:
# Instalar dependencias si es necesario
# %pip install skforecast tensorflow scikit-learn plotly statsmodels seaborn

In [ ]:
# ── Datos y matemáticas ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualización ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "plotly_white"
poff.init_notebook_mode(connected=True)

# ── Preprocesamiento ──────────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                              r2_score)

# ── Deep Learning ─────────────────────────────────────────────────────────────
from keras.models import Sequential
from keras.layers import (Input, LSTM, GRU, Dense, Dropout,
                           Conv1D, GlobalAveragePooling1D, Flatten)
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

# ── Series de tiempo ──────────────────────────────────────────────────────────
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

print("Librerías cargadas correctamente.")

## 4. Datos

Los datos provienen de la **Red de Vigilancia y Control de la Contaminación
Atmosférica** de la Generalitat Valenciana, estación
*46250054 – València Centre*.

| Característica | Valor |
|---|---|
| Período | 2019-01-01 → 2023-12-31 |
| Frecuencia | Horaria (1 registro/hora) |
| Total de registros | 43 824 horas |
| Variables | SO₂, CO, NO, NO₂, PM10, NOx, O₃, velocidad del viento, dirección del viento, PM2.5 |

> **Variable objetivo:** Ozono (O₃) en µg/m³.
> El O₃ tiene un ciclo diurno muy marcado: mínimo nocturno y máximo al mediodía
> por la fotoquímica solar, lo que lo hace ideal para ilustrar estacionalidad
> múltiple con redes neuronales.

In [ ]:
# ── Carga del dataset ─────────────────────────────────────────────────────────
URL = ("https://github.com/Wilsonsr/Series-de-Tiempo/blob/main/"
       "Data/air_quality_valencia_no_missing.csv?raw=true")

air_quality_raw = pd.read_csv(URL, sep=",")

# Convertir datetime ANTES de set_index (orden correcto)
air_quality_raw['datetime'] = pd.to_datetime(air_quality_raw['datetime'])
air_quality = air_quality_raw.set_index('datetime').copy()

# Establecer frecuencia horaria explícitamente
air_quality.index.freq = pd.tseries.frequencies.to_offset('h')

print(f"Forma del dataset : {air_quality.shape}")
print(f"Período           : {air_quality.index.min()} → {air_quality.index.max()}")
print(f"Frecuencia        : {air_quality.index.freq}")
print(f"Columnas          : {list(air_quality.columns)}")
air_quality.head()

In [ ]:
# ── Vista rápida de los últimos registros ─────────────────────────────────────
air_quality.tail()

## 5. Análisis Exploratorio de Datos (EDA)

> **¿Por qué el EDA es obligatorio antes de construir un modelo?**
>
> Sin EDA, corremos el riesgo de modelar ruido, ignorar valores atípicos que
> distorsionan el entrenamiento, o elegir una arquitectura inadecuada.
> El EDA responde preguntas críticas: ¿hay valores faltantes? ¿qué patrones
> existen? ¿cuáles variables están relacionadas con el objetivo?

### 5.1 Valores faltantes

In [ ]:
# ── Verificación de valores faltantes ────────────────────────────────────────
print("=== VALORES FALTANTES ===")
missing = air_quality.isnull().sum()
if missing.any():
    print(missing[missing > 0])
else:
    print("El dataset no tiene valores faltantes.")

# Verificar continuidad temporal (sin huecos)
fechas_esperadas = pd.date_range(
    start=air_quality.index.min(),
    end=air_quality.index.max(),
    freq='h'
)
huecos = fechas_esperadas.difference(air_quality.index)
print(f"\nRegistros esperados : {len(fechas_esperadas):,}")
print(f"Registros presentes : {len(air_quality):,}")
print(f"Horas faltantes     : {len(huecos)}")

### 5.2 Estadísticos descriptivos

In [ ]:
# ── Estadísticos descriptivos ────────────────────────────────────────────────
desc = air_quality.describe().round(2)
print(desc.to_string())

### 5.3 Visualización de las series principales

In [ ]:
# ── Series con media móvil semanal (168 h) ───────────────────────────────────
variables_plot = ['o3', 'pm2.5', 'no2', 'pm10', 'co', 'so2']
unidades = {'o3': 'µg/m³', 'pm2.5': 'µg/m³', 'no2': 'µg/m³',
            'pm10': 'µg/m³', 'co': 'mg/m³', 'so2': 'µg/m³'}

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
colores = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b']

for ax, var, color in zip(axes.flatten(), variables_plot, colores):
    raw = air_quality[var]
    ma  = raw.rolling(24 * 7, center=True).mean()
    ax.plot(raw.resample('D').mean(), color=color, alpha=0.3, linewidth=0.6,
            label='Media diaria')
    ax.plot(ma.resample('D').mean(), color=color, linewidth=1.8,
            label='Media móvil 7 días')
    ax.set_title(f'{var.upper()} ({unidades.get(var, "")})', fontsize=11)
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle('Contaminantes del aire — Valencia 2019–2023', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 5.4 Patrones estacionales del O₃

In [ ]:
# ── Patrón intradiario y semanal del O₃ ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Patrón horario
o3_hora = air_quality.groupby(air_quality.index.hour)['o3'].agg(['mean', 'std'])
axes[0].fill_between(o3_hora.index,
                     o3_hora['mean'] - o3_hora['std'],
                     o3_hora['mean'] + o3_hora['std'],
                     alpha=0.25, color='darkorange')
axes[0].plot(o3_hora.index, o3_hora['mean'], marker='o',
             color='darkorange', linewidth=2)
axes[0].set_title('Patrón intradiario de O₃\n(promedio ± 1 desv. estándar)', fontsize=11)
axes[0].set_xlabel('Hora del día')
axes[0].set_ylabel('O₃ (µg/m³)')
axes[0].set_xticks(range(0, 24, 3))
axes[0].grid(True, alpha=0.3)

# Patrón semanal
dias = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
o3_dia = air_quality.groupby(air_quality.index.dayofweek)['o3'].mean()
axes[1].bar(range(7), o3_dia.values, color='teal', edgecolor='black', alpha=0.8)
axes[1].set_title('Patrón semanal de O₃', fontsize=11)
axes[1].set_xlabel('Día de la semana')
axes[1].set_ylabel('O₃ promedio (µg/m³)')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dias)
axes[1].grid(axis='y', alpha=0.3)

# Patrón mensual
meses = ['Ene','Feb','Mar','Abr','May','Jun',
         'Jul','Ago','Sep','Oct','Nov','Dic']
o3_mes = air_quality.groupby(air_quality.index.month)['o3'].mean()
axes[2].bar(range(1, 13), o3_mes.values, color='mediumvioletred',
            edgecolor='black', alpha=0.8)
axes[2].set_title('Patrón mensual de O₃', fontsize=11)
axes[2].set_xlabel('Mes')
axes[2].set_ylabel('O₃ promedio (µg/m³)')
axes[2].set_xticks(range(1, 13))
axes[2].set_xticklabels(meses, rotation=45)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Estacionalidad múltiple del Ozono (O₃)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Observaciones clave:")
print(f"  Hora pico O₃   : {o3_hora['mean'].idxmax():02d}:00 h  ({o3_hora['mean'].max():.1f} µg/m³)")
print(f"  Hora mínima O₃ : {o3_hora['mean'].idxmin():02d}:00 h  ({o3_hora['mean'].min():.1f} µg/m³)")
print(f"  Mes pico O₃    : {meses[o3_mes.idxmax()-1]}  ({o3_mes.max():.1f} µg/m³)")

### 5.5 Correlación entre variables

In [ ]:
# ── Matriz de correlación ─────────────────────────────────────────────────────
# Usamos correlación de Spearman (robusta a outliers y no-normalidad)
corr = air_quality.corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, square=True, linewidths=0.5,
            vmin=-1, vmax=1, annot_kws={'size': 9})
ax.set_title('Correlación de Spearman entre contaminantes y variables meteorológicas',
             fontsize=12)
plt.tight_layout()
plt.show()

# Correlaciones con O₃
print("\nCorrelaciones con O₃ (Spearman):")
print(corr['o3'].drop('o3').sort_values(ascending=False).to_string())

### 5.6 Detección de outliers

In [ ]:
# ── Outliers por método IQR (3×IQR = extremos) ───────────────────────────────
print("=== OUTLIERS EXTREMOS (|x - mediana| > 3×IQR) ===")
print(f"{'Variable':10s}  {'N outliers':>12s}  {'% del total':>12s}  {'Máx valor':>10s}")
print("-" * 52)
for col in air_quality.columns:
    Q1  = air_quality[col].quantile(0.25)
    Q3  = air_quality[col].quantile(0.75)
    IQR = Q3 - Q1
    mask_out = (air_quality[col] < Q1 - 3*IQR) | (air_quality[col] > Q3 + 3*IQR)
    n   = mask_out.sum()
    pct = 100 * n / len(air_quality)
    mx  = air_quality[col].max()
    print(f"  {col:10s}: {n:10d}    {pct:10.2f}%   {mx:10.1f}")

# Box-plot de todas las variables (valores escalados para comparar)
from sklearn.preprocessing import StandardScaler
sc_temp = StandardScaler()
df_std = pd.DataFrame(sc_temp.fit_transform(air_quality),
                      columns=air_quality.columns)

fig, ax = plt.subplots(figsize=(12, 5))
df_std.boxplot(ax=ax, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='navy'),
               medianprops=dict(color='red', linewidth=2))
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('Distribución de variables (estandarizadas) — detección de outliers',
             fontsize=11)
ax.set_ylabel('Valor estandarizado (z-score)')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### 5.7 ACF y PACF del O₃

In [ ]:
# ── ACF y PACF para analizar autocorrelación temporal ────────────────────────
# Usamos media diaria para mayor claridad visual
o3_diario = air_quality['o3'].resample('D').mean().dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(o3_diario, lags=60, ax=axes[0],
         title='ACF — O₃ diario (lags en días)')
plot_pacf(o3_diario, lags=60, ax=axes[1],
          title='PACF — O₃ diario (lags en días)', method='ywm')
plt.tight_layout()
plt.show()

print('''
Interpretacion:
  ACF  : decae lentamente -> hay autocorrelacion positiva persistente (tendencia/ciclo).
         Pico en lag ~7 confirma patron semanal.
  PACF : corte abrupto despues de los primeros lags -> los modelos AR de orden bajo
         pueden capturar buena parte de la estructura.
  Para la LSTM: ventanas de 24-48 horas suelen ser suficientes para el patron
  intradiario; ventanas de 7 dias (168 h) para capturar el ciclo semanal.
''')

### 5.8 Nota sobre la dirección del viento (variable circular)

> **Atención metodológica:** La variable `direc.` (dirección del viento en grados,
> 0°–360°) es **circular**: 0° y 360° son el mismo ángulo, pero el MinMaxScaler
> las trataría como los extremos opuestos del rango. Esto introduce un error
> sistemático en el modelo.
>
> La solución correcta es descomponerla en sus **componentes seno y coseno**:
>
> ```
> viento_sin = sin(direc × π/180)   → componente Norte-Sur
> viento_cos = cos(direc × π/180)   → componente Este-Oeste
> ```
>
> Así, 350° y 10° quedan cerca en el espacio transformado.

In [ ]:
# ── Transformación de variable circular ──────────────────────────────────────
air_quality['viento_sin'] = np.sin(np.radians(air_quality['direc.']))
air_quality['viento_cos'] = np.cos(np.radians(air_quality['direc.']))
air_quality = air_quality.drop(columns=['direc.'])

print("Variables después de la transformación circular:")
print(list(air_quality.columns))

## 6. División Train / Validación / Test

### ¿Por qué tres conjuntos y no solo dos?

| Conjunto | Rol | Analogía |
|----------|-----|----------|
| **Train** | El modelo aprende de estos datos | Estudiar para el examen |
| **Validación** | Ajustar hiperparámetros y detectar sobreajuste | Simulacros de práctica |
| **Test** | Evaluación final, nunca vista durante el entrenamiento | El examen real |

> **Regla de oro en series de tiempo:** el orden temporal es sagrado.
> Los datos de train deben ser **anteriores** a los de validación, y estos
> **anteriores** a los de test. Mezclar fechas introduce *fuga de información*
> (data leakage) y produce evaluaciones optimistas y falsas.

### Evitar el solape en los cortes

Con `pandas.loc`, ambos extremos son **inclusivos**. Si el punto de corte
está en dos conjuntos a la vez, el mismo dato se usa para ajustar el modelo
(train/val) y también para evaluarlo, lo que contamina la evaluación.

```python
# MAL: el timestamp de corte queda en AMBOS conjuntos
train = df.loc[:end_train]         # incluye end_train
val   = df.loc[end_train:end_val]  # también incluye end_train  ← duplicado

# BIEN: usar iloc[1:] para saltar el punto duplicado
val   = df.loc[end_train:end_val].iloc[1:]  # excluye el primer punto
```

In [ ]:
# ── División temporal sin solape ─────────────────────────────────────────────
end_train = '2022-12-31 23:00:00'   # ~80% de los datos
end_val   = '2023-06-30 23:00:00'   # ~10% siguientes

aq_train = air_quality.loc[:end_train].copy()
aq_val   = air_quality.loc[end_train:end_val].iloc[1:].copy()   # salta duplicado
aq_test  = air_quality.loc[end_val:].iloc[1:].copy()             # salta duplicado

for nombre, df in [('Train     ', aq_train),
                   ('Validación', aq_val),
                   ('Test      ', aq_test)]:
    pct = 100 * len(df) / len(air_quality)
    print(f"  {nombre}: {df.index.min().date()} → {df.index.max().date()}"
          f"  n={len(df):,}  ({pct:.1f}%)")

In [ ]:
# ── Visualización del split con media móvil semanal ──────────────────────────
window = 24 * 7   # 7 días

fig = go.Figure()
for nombre, df, color in [('Entrenamiento', aq_train, '#1f77b4'),
                           ('Validación',   aq_val,   '#ff7f0e'),
                           ('Test',         aq_test,  '#2ca02c')]:
    ma = df['o3'].rolling(window, center=True).mean()
    fig.add_trace(go.Scatter(
        x=ma.index, y=ma, name=nombre,
        line=dict(color=color, width=2)
    ))

fig.update_layout(
    title='Concentración de O₃ — Media móvil 7 días (división train/val/test)',
    xaxis_title='Fecha', yaxis_title='O₃ (µg/m³)',
    height=420, template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.show()

## 7. Escalado sin fuga de información

Las redes neuronales son sensibles a la **magnitud de los valores**: si una
variable está en el rango [0, 500] y otra en [0, 1], la primera domina el
gradiente durante el entrenamiento.

El **MinMaxScaler** normaliza cada variable al rango [0, 1]:

$$x_{\text{esc}} = \frac{x - x_{\min}^{\text{train}}}{x_{\max}^{\text{train}} - x_{\min}^{\text{train}}}$$

### ¿Por qué usar sólo los datos de train para ajustar el scaler?

- Si usamos validación o test para calcular `min` y `max`, estamos usando
  información **del futuro** para transformar el pasado → *data leakage*.
- El scaler ajustado en train se **aplica** (`.transform`) en val y test,
  usando los mismos parámetros del train.

> Puede ocurrir que algunos valores de val/test estén fuera del rango de train
> (e.g., un pico de contaminación excepcional). En ese caso el valor escalado
> será > 1 o < 0, lo cual es aceptable y no introduce sesgo.

In [ ]:
# ── Escalado univariado para el modelo LSTM base (O₃) ────────────────────────
TARGET = 'o3'

train_serie = aq_train[[TARGET]].copy()
val_serie   = aq_val[[TARGET]].copy()
test_serie  = aq_test[[TARGET]].copy()

scaler_uni = MinMaxScaler(feature_range=(0, 1))
train_sc = scaler_uni.fit_transform(train_serie)    # fit + transform en train
val_sc   = scaler_uni.transform(val_serie)           # solo transform
test_sc  = scaler_uni.transform(test_serie)          # solo transform

print(f"Min O₃ train: {train_serie.min().values[0]:.1f}  →  escalado: {train_sc.min():.4f}")
print(f"Max O₃ train: {train_serie.max().values[0]:.1f}  →  escalado: {train_sc.max():.4f}")
print(f"Valores en val fuera de [0,1]: {((val_sc < 0) | (val_sc > 1)).sum()}")
print(f"Valores en test fuera de [0,1]: {((test_sc < 0) | (test_sc > 1)).sum()}")

In [ ]:
# ── Visualización de la serie escalada ───────────────────────────────────────
train_sc_df = pd.DataFrame(train_sc, index=train_serie.index, columns=[TARGET])

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
train_serie.rolling(24*7).mean().plot(ax=axes[0], color='steelblue')
axes[0].set_title('O₃ original (µg/m³) — media móvil 7 días')
axes[0].set_xlabel('Fecha')
axes[0].set_ylabel('O₃ (µg/m³)')
axes[0].grid(True, alpha=0.3)

train_sc_df.rolling(24*7).mean().plot(ax=axes[1], color='darkorange')
axes[1].set_title('O₃ escalado [0,1] — media móvil 7 días')
axes[1].set_xlabel('Fecha')
axes[1].set_ylabel('Valor escalado')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Construcción de ventanas deslizantes (lag features)

Los modelos LSTM no reciben la serie de tiempo directamente; necesitan que
los datos se organicen en pares **(X, y)** donde:

- **X** = los últimos `lags` valores (la "ventana de contexto").
- **y** = el o los valores que el modelo debe predecir (el "objetivo").

```
Serie original:  [v1, v2, v3, v4, v5, v6, v7, ...]

lags = 3, steps = 1:
  X[0] = [v1, v2, v3]  →  y[0] = v4
  X[1] = [v2, v3, v4]  →  y[1] = v5
  X[2] = [v3, v4, v5]  →  y[2] = v6
  ...

lags = 3, steps = 2:
  X[0] = [v1, v2, v3]  →  y[0] = [v4, v5]
  X[1] = [v2, v3, v4]  →  y[1] = [v5, v6]
  ...
```

**Formato para LSTM:** `X` debe tener 3 dimensiones: `(n_muestras, lags, n_features)`.

In [ ]:
# ── Función unificada de creación de secuencias ───────────────────────────────
def crear_secuencias(data: np.ndarray, lags: int, steps: int = 1) -> tuple:
    """
    Transforma un array temporal en pares (X, y) para aprendizaje supervisado.

    Parámetros
    ----------
    data   : array 2D (n_obs, n_features) — ya escalado
    lags   : número de pasos pasados como contexto
    steps  : número de pasos futuros a predecir

    Retorna
    -------
    X : (n_muestras, lags, n_features)
    y : (n_muestras, steps)  — solo la columna objetivo (índice 0 del array)
    """
    X, y = [], []
    for i in range(len(data) - lags - steps + 1):
        X.append(data[i : i + lags])
        y.append(data[i + lags : i + lags + steps, 0])  # columna 0 = objetivo
    return np.array(X), np.array(y)


# ── Parámetros del modelo ─────────────────────────────────────────────────────
LAGS  = 48    # 48 horas de contexto (2 días): captura el ciclo diurno × 2
STEPS = 1     # predecir la siguiente hora

X_train, y_train = crear_secuencias(train_sc, LAGS, STEPS)
X_val,   y_val   = crear_secuencias(val_sc,   LAGS, STEPS)
X_test,  y_test  = crear_secuencias(test_sc,  LAGS, STEPS)

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}    y_test  : {y_test.shape}")

In [ ]:
# ── Visualización de una ventana de ejemplo ───────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))

# La secuencia de entrada (últimas LAGS horas)
ax.plot(range(LAGS), X_train[0, :, 0], color='steelblue',
        linewidth=1.5, marker='.', markersize=4, label='Contexto (lags)')

# El valor que el modelo debe predecir
ax.scatter(LAGS, y_train[0, 0], color='red', s=100, zorder=5,
           label=f'Valor objetivo (t+1)')

ax.axvline(LAGS - 0.5, color='gray', linestyle='--', linewidth=1)
ax.set_title(f'Ejemplo de ventana deslizante (lags={LAGS})', fontsize=12)
ax.set_xlabel('Pasos temporales anteriores')
ax.set_ylabel('O₃ escalado [0, 1]')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Modelo LSTM Univariado — Pronóstico 1 paso (1 hora)

Este es el modelo más simple: usa sólo los valores pasados de O₃ para
predecir el valor de la siguiente hora.

### Arquitectura

```
Input: (batch, 48, 1)   ← 48 horas de O₃ escalado
LSTM(64)                ← aprende dependencias temporales
Dropout(0.2)            ← regularización: apaga 20% de neuronas al azar
Dense(32, relu)         ← capa intermedia no lineal
Dense(1)                ← predicción de la siguiente hora
```

### Decisiones de diseño

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| `lags` | 48 h | Captura 2 ciclos diurnos completos |
| `units` LSTM | 64 | Balance entre capacidad y velocidad |
| `learning_rate` | 0.001 | Estándar para LSTM; 0.01 suele oscilar |
| `batch_size` | 64 | Suficientemente grande para gradientes estables |
| `patience` | 15 | Permite convergencia sin cortar demasiado pronto |
| `restore_best_weights` | True | Guarda el mejor modelo, no el último |

In [ ]:
# ── Arquitectura LSTM — estilo Keras moderno (sin warning de input_shape) ─────
model_lstm = Sequential([
    Input(shape=(LAGS, 1)),
    LSTM(units=64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(STEPS)
], name='LSTM_univariado')

model_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse'
)
model_lstm.summary()

In [ ]:
# ── Entrenamiento con EarlyStopping correcto ─────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,    # guarda los pesos del MEJOR epoch, no el último
    verbose=1
)

history_lstm = model_lstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# ── Curvas de pérdida ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_lstm.history['loss'],     label='Pérdida entrenamiento', color='steelblue')
ax.plot(history_lstm.history['val_loss'], label='Pérdida validación',    color='darkorange')

best_epoch = np.argmin(history_lstm.history['val_loss'])
ax.axvline(best_epoch, color='red', linestyle='--', linewidth=1.2,
           label=f'Mejor epoch ({best_epoch+1})')

ax.set_title('Curvas de pérdida (MSE) — LSTM univariado', fontsize=12)
ax.set_xlabel('Época')
ax.set_ylabel('Error Cuadrático Medio (MSE)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mejor val_loss: {min(history_lstm.history['val_loss']):.6f}  (época {best_epoch+1})")
print(f"Entrenó {len(history_lstm.history['loss'])} épocas (detenido por EarlyStopping)")

In [ ]:
# ── Función de evaluación reutilizable ───────────────────────────────────────
def evaluar_modelo(modelo, X_test, y_test, scaler, nombre='Modelo'):
    """Predice, desescala y calcula métricas completas."""
    y_pred_sc = modelo.predict(X_test, verbose=0)
    y_pred = scaler.inverse_transform(y_pred_sc)
    y_real = scaler.inverse_transform(y_test)

    rmse  = np.sqrt(mean_squared_error(y_real, y_pred))
    mae   = mean_absolute_error(y_real, y_pred)
    r2    = r2_score(y_real, y_pred)
    smape = float(np.mean(2 * np.abs(y_pred - y_real) /
                          (np.abs(y_real) + np.abs(y_pred) + 1e-8)) * 100)

    print(f"{'='*48}")
    print(f"  {nombre}")
    print(f"{'='*48}")
    print(f"  RMSE  (raíz del MSE)            : {rmse:.3f} µg/m³")
    print(f"  MAE   (error absoluto medio)     : {mae:.3f} µg/m³")
    print(f"  R²    (coef. de determinación)   : {r2:.4f}")
    print(f"  SMAPE (error porcentual simétrico): {smape:.2f}%")
    print(f"{'='*48}")
    print(f"  Media O₃ en test                 : {y_real.mean():.2f} µg/m³")
    print(f"  RMSE como % de la media          : {100*rmse/y_real.mean():.1f}%")
    return y_real, y_pred, {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'SMAPE': smape}


y_real_lstm, y_pred_lstm, metricas_lstm = evaluar_modelo(
    model_lstm, X_test, y_test, scaler_uni, 'LSTM univariado 1-paso'
)

In [ ]:
# ── Comparación con baseline de persistencia ─────────────────────────────────
# Baseline: "el valor del siguiente período es igual al actual"
# Es el modelo más simple posible; si la LSTM no lo supera, no aporta valor.

y_naive = y_real_lstm[:-1]
y_naive_tgt = y_real_lstm[1:]
rmse_naive = np.sqrt(mean_squared_error(y_naive_tgt, y_naive))

print(f"RMSE Baseline (persistencia) : {rmse_naive:.3f} µg/m³")
print(f"RMSE LSTM                    : {metricas_lstm['RMSE']:.3f} µg/m³")
print(f"Mejora sobre baseline        : {100*(1 - metricas_lstm['RMSE']/rmse_naive):.1f}%")

In [ ]:
# ── Visualización de predicciones vs real ────────────────────────────────────
index_test = test_serie.index[LAGS: LAGS + len(y_real_lstm)]
df_res = pd.DataFrame({
    'Real':     y_real_lstm.flatten(),
    'Predicho': y_pred_lstm.flatten()
}, index=index_test)

# Gráfico interactivo (últimas 2 semanas para mayor detalle)
df_zoom = df_res.tail(24 * 14)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_zoom.index, y=df_zoom['Real'],
    name='Real', line=dict(color='black', width=1.5)
))
fig.add_trace(go.Scatter(
    x=df_zoom.index, y=df_zoom['Predicho'],
    name='Predicho LSTM', line=dict(color='red', width=1.5, dash='dash')
))
fig.update_layout(
    title='Predicción de O₃ — LSTM univariado (últimas 2 semanas del test)',
    xaxis_title='Fecha', yaxis_title='O₃ (µg/m³)',
    template='plotly_white', height=420,
    xaxis=dict(rangeslider=dict(visible=True), type='date'),
    legend=dict(x=0.01, y=0.99)
)
fig.show()

In [ ]:
# ── Análisis de residuales ────────────────────────────────────────────────────
residuos = df_res['Real'] - df_res['Predicho']

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histograma de errores
axes[0].hist(residuos, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Distribución de residuales')
axes[0].set_xlabel('Error (Real - Predicho) en µg/m³')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(True, alpha=0.3)

# Residuales vs predicho
axes[1].scatter(df_res['Predicho'], residuos, alpha=0.2, s=5, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuales vs valores predichos')
axes[1].set_xlabel('Valor predicho (µg/m³)')
axes[1].set_ylabel('Residual (µg/m³)')
axes[1].grid(True, alpha=0.3)

# Real vs predicho
axes[2].scatter(df_res['Real'], df_res['Predicho'], alpha=0.2, s=5, color='steelblue')
mn = min(df_res.min())
mx = max(df_res.max())
axes[2].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Predicción perfecta')
axes[2].set_title('Real vs Predicho')
axes[2].set_xlabel('Real (µg/m³)')
axes[2].set_ylabel('Predicho (µg/m³)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Diagnóstico de residuales — LSTM univariado', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Media de residuales  : {residuos.mean():.4f} (idealmente ~0)")
print(f"Desv. estándar       : {residuos.std():.4f}")

In [ ]:
# ── Pronóstico fuera de muestra (función corregida) ───────────────────────────
def pronostico_futuro(modelo, data_sc: np.ndarray, lags: int,
                      pasos: int, scaler, index_original) -> pd.DataFrame:
    """
    Pronóstico autoregresivo: usa cada predicción como entrada del siguiente paso.
    Adecuado para visualizar el comportamiento del modelo más allá del test.

    NOTA: el error se acumula con cada paso (error compounding).
    Para pronósticos largos, preferir el enfoque directo (Dense(steps)).
    """
    secuencia = data_sc[-lags:].reshape(1, lags, 1).copy()
    preds = []
    for _ in range(pasos):
        p = modelo.predict(secuencia, verbose=0)[0, 0]
        preds.append(p)
        secuencia = np.roll(secuencia, -1, axis=1)
        secuencia[0, -1, 0] = p

    preds_real = scaler.inverse_transform(
        np.array(preds).reshape(-1, 1)
    ).flatten()

    freq_str = 'h'
    fechas = pd.date_range(
        start=index_original[-1] + pd.Timedelta(hours=1),
        periods=pasos,
        freq=freq_str
    )
    return pd.DataFrame({'Pronostico_O3': preds_real}, index=fechas)


df_forecast = pronostico_futuro(
    model_lstm, test_sc, lags=LAGS, pasos=72,   # 72 h = 3 días
    scaler=scaler_uni, index_original=test_serie.index
)

# Combinar con datos reales recientes para contexto
ultimas_reales = test_serie.tail(7 * 24)   # última semana real

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ultimas_reales.index, y=ultimas_reales['o3'],
    name='Histórico real', line=dict(color='black', width=2)
))
fig.add_trace(go.Scatter(
    x=df_forecast.index, y=df_forecast['Pronostico_O3'],
    name='Pronóstico 72 h', line=dict(color='red', width=2, dash='dot'),
    fill='tonexty', fillcolor='rgba(255,0,0,0.05)'
))
fig.add_vrect(
    x0=df_forecast.index[0], x1=df_forecast.index[-1],
    fillcolor='red', opacity=0.05, line_width=0
)
fig.update_layout(
    title='Pronóstico fuera de muestra — O₃ (72 horas = 3 días)',
    xaxis_title='Fecha', yaxis_title='O₃ (µg/m³)',
    template='plotly_white', height=420, legend=dict(x=0.01, y=0.99)
)
fig.show()

## 10. LSTM Univariado — Multi-step (Horizonte variable)

### Enfoque directo vs autoregresivo

| Enfoque | Cómo funciona | Ventaja | Desventaja |
|---------|--------------|---------|------------|
| **Directo** (`Dense(H)`) | Predice todos los H pasos en una sola pasada | Sin acumulación de errores | El modelo debe reentrenarse si cambia H |
| **Autoregresivo** (`Dense(1)` + bucle) | Predice 1 paso, lo usa como entrada del siguiente | Flexible para cualquier H | El error se acumula paso a paso |

En esta sección usamos el **enfoque directo**: la capa de salida tiene `steps` neuronas,
una por cada paso del horizonte.

### ¿Cómo cambia el error con el horizonte?

Se espera que el MSE aumente con el horizonte: predecir 24 horas adelante es más
difícil que predecir 1 hora. Esta sección lo verifica experimentalmente.

In [ ]:
# ── Experimento multi-step: comparar horizontes H ────────────────────────────
LAGS_MS  = 48    # contexto fijo
HORIZONTES = [6, 12, 24, 48, 72]

resultados_ms = {}

for H in HORIZONTES:
    print(f"Entrenando LSTM directo con H={H} pasos...", end=' ')

    X_tr, y_tr = crear_secuencias(train_sc, LAGS_MS, H)
    X_vl, y_vl = crear_secuencias(val_sc,   LAGS_MS, H)
    X_te, y_te = crear_secuencias(test_sc,  LAGS_MS, H)

    modelo_ms = Sequential([
        Input(shape=(LAGS_MS, 1)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dense(H)
    ], name=f'LSTM_direct_H{H}')

    modelo_ms.compile(optimizer=Adam(0.001), loss='mse')
    modelo_ms.fit(
        X_tr, y_tr,
        validation_data=(X_vl, y_vl),
        epochs=80, batch_size=64,
        callbacks=[EarlyStopping(monitor='val_loss', patience=10,
                                 restore_best_weights=True)],
        verbose=0
    )

    y_pred_sc = modelo_ms.predict(X_te, verbose=0)
    y_te_inv  = scaler_uni.inverse_transform(y_te)
    y_pred_inv = scaler_uni.inverse_transform(y_pred_sc)

    rmse = np.sqrt(mean_squared_error(y_te_inv.flatten(), y_pred_inv.flatten()))
    mae  = mean_absolute_error(y_te_inv.flatten(), y_pred_inv.flatten())
    resultados_ms[H] = {
        'RMSE': rmse, 'MAE': mae,
        'y_real': y_te_inv, 'y_pred': y_pred_inv
    }
    print(f"RMSE={rmse:.2f} µg/m³")

In [ ]:
# ── Visualización: RMSE vs horizonte ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# RMSE vs horizonte
rmse_vals = [resultados_ms[H]['RMSE'] for H in HORIZONTES]
axes[0].plot(HORIZONTES, rmse_vals, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('RMSE según horizonte de predicción', fontsize=12)
axes[0].set_xlabel('Horizonte H (horas)')
axes[0].set_ylabel('RMSE (µg/m³)')
axes[0].grid(True, alpha=0.3)
for H, r in zip(HORIZONTES, rmse_vals):
    axes[0].annotate(f'{r:.1f}', (H, r), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=9)

# Predicción para H=24 (primera secuencia del test)
H_ejemplo = 24
yr = resultados_ms[H_ejemplo]['y_real'][0]
yp = resultados_ms[H_ejemplo]['y_pred'][0]
axes[1].plot(yr, label='Real', color='black', linewidth=2)
axes[1].plot(yp, label='Predicho', color='red', linestyle='--', linewidth=2)
axes[1].fill_between(range(H_ejemplo), yr, yp, alpha=0.2, color='red')
axes[1].set_title(f'Ejemplo de predicción H={H_ejemplo} horas', fontsize=12)
axes[1].set_xlabel('Paso del horizonte')
axes[1].set_ylabel('O₃ (µg/m³)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('LSTM Directo — Multi-step forecasting', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Tabla resumen multi-step ──────────────────────────────────────────────────
tabla_ms = pd.DataFrame({
    H: {'RMSE (µg/m³)': resultados_ms[H]['RMSE'],
        'MAE (µg/m³)' : resultados_ms[H]['MAE']}
    for H in HORIZONTES
}).T.round(3)
tabla_ms.index.name = 'Horizonte H'
print(tabla_ms.to_string())

## 11. Modelo LSTM Multivariado (N:1)

### ¿Cuándo usar variables exógenas?

En el modelo univariado, el O₃ se predice solo a partir de su propio historial.
Pero en la atmósfera, el O₃ es el producto de reacciones fotoquímicas que involucran
NO₂, CO, radiación solar, temperatura y viento. Por lo tanto, usar **variables
exógenas** (NO₂, PM2.5, CO, etc.) puede mejorar las predicciones.

### Consideraciones para el modelo N:1

1. **Cada variable debe escalarse con su propio scaler** ajustado en train.
2. La función de secuencias ahora recibe un array 2D `(tiempo, n_features)`.
3. La variable circular `direc.` ya fue transformada en `viento_sin`/`viento_cos`.
4. El desescalado de las predicciones usa **solo el scaler de O₃**.

In [ ]:
# ── Configuración multivariada ────────────────────────────────────────────────
FEATURES = ['pm2.5', 'co', 'no', 'no2', 'pm10', 'nox',
            'o3', 'veloc.', 'so2', 'viento_sin', 'viento_cos']
TARGET_MV = 'o3'
LAGS_MV   = 48
STEPS_MV  = 5      # predecir las próximas 5 horas

# Separar train/val/test con todas las features
aq_train_mv = aq_train[FEATURES].copy()
aq_val_mv   = aq_val[FEATURES].copy()
aq_test_mv  = aq_test[FEATURES].copy()

# Escalar cada variable con su propio scaler (fit solo en train)
scalers_mv = {}
train_mv_sc = np.zeros_like(aq_train_mv.values, dtype=float)
val_mv_sc   = np.zeros_like(aq_val_mv.values,   dtype=float)
test_mv_sc  = np.zeros_like(aq_test_mv.values,  dtype=float)

for j, col in enumerate(FEATURES):
    sc = MinMaxScaler()
    train_mv_sc[:, j] = sc.fit_transform(aq_train_mv[[col]]).flatten()
    val_mv_sc[:,   j] = sc.transform(aq_val_mv[[col]]).flatten()
    test_mv_sc[:,  j] = sc.transform(aq_test_mv[[col]]).flatten()
    scalers_mv[col]   = sc

idx_target = FEATURES.index(TARGET_MV)
print(f"Número de features   : {len(FEATURES)}")
print(f"Índice de '{TARGET_MV}' : {idx_target}")
print(f"Shape train escalado : {train_mv_sc.shape}")

In [ ]:
# ── Crear secuencias multivariadas ────────────────────────────────────────────
def crear_seq_multivariado(data_2d: np.ndarray, idx_target: int,
                            lags: int, steps: int) -> tuple:
    """
    data_2d : (n_obs, n_features) — ya escalado
    Retorna X: (n, lags, n_features)  y: (n, steps)
    """
    X, y = [], []
    for i in range(len(data_2d) - lags - steps + 1):
        X.append(data_2d[i : i + lags, :])
        y.append(data_2d[i + lags : i + lags + steps, idx_target])
    return np.array(X), np.array(y)


X_tr_mv, y_tr_mv = crear_seq_multivariado(train_mv_sc, idx_target, LAGS_MV, STEPS_MV)
X_vl_mv, y_vl_mv = crear_seq_multivariado(val_mv_sc,   idx_target, LAGS_MV, STEPS_MV)
X_te_mv, y_te_mv = crear_seq_multivariado(test_mv_sc,  idx_target, LAGS_MV, STEPS_MV)

print(f"X_train_mv : {X_tr_mv.shape}   y_train_mv : {y_tr_mv.shape}")
print(f"X_val_mv   : {X_vl_mv.shape}     y_val_mv   : {y_vl_mv.shape}")
print(f"X_test_mv  : {X_te_mv.shape}    y_test_mv  : {y_te_mv.shape}")

In [ ]:
# ── Arquitectura LSTM multivariada (2 capas) ─────────────────────────────────
n_feat = len(FEATURES)

model_mv = Sequential([
    Input(shape=(LAGS_MV, n_feat)),
    LSTM(128, return_sequences=True),
    Dropout(0.25),
    LSTM(64),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(STEPS_MV)
], name='LSTM_multivariado')

model_mv.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
model_mv.summary()

In [ ]:
# ── Entrenamiento multivariado ────────────────────────────────────────────────
hist_mv = model_mv.fit(
    X_tr_mv, y_tr_mv,
    validation_data=(X_vl_mv, y_vl_mv),
    epochs=100, batch_size=64,
    callbacks=[EarlyStopping(monitor='val_loss', patience=15,
                             restore_best_weights=True, verbose=1)],
    verbose=1
)

In [ ]:
# ── Curvas de pérdida ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist_mv.history['loss'],     label='Entrenamiento', color='steelblue')
ax.plot(hist_mv.history['val_loss'], label='Validación',    color='darkorange')
best = np.argmin(hist_mv.history['val_loss'])
ax.axvline(best, color='red', linestyle='--', label=f'Mejor época ({best+1})')
ax.set_title('Curvas de pérdida — LSTM multivariado', fontsize=12)
ax.set_xlabel('Época')
ax.set_ylabel('MSE')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Predicción y evaluación ───────────────────────────────────────────────────
sc_target = scalers_mv[TARGET_MV]

y_pred_mv_sc = model_mv.predict(X_te_mv, verbose=0)
y_te_mv_inv  = sc_target.inverse_transform(y_te_mv)
y_pred_mv_inv = sc_target.inverse_transform(y_pred_mv_sc)

rmse_mv = np.sqrt(mean_squared_error(y_te_mv_inv.flatten(), y_pred_mv_inv.flatten()))
mae_mv  = mean_absolute_error(y_te_mv_inv.flatten(), y_pred_mv_inv.flatten())
r2_mv   = r2_score(y_te_mv_inv.flatten(), y_pred_mv_inv.flatten())

print(f"LSTM Multivariado ({len(FEATURES)} features → O₃, {STEPS_MV} pasos)")
print(f"  RMSE : {rmse_mv:.3f} µg/m³")
print(f"  MAE  : {mae_mv:.3f} µg/m³")
print(f"  R²   : {r2_mv:.4f}")
print(f"\nLSTM Univariado 1-paso (referencia):")
print(f"  RMSE : {metricas_lstm['RMSE']:.3f} µg/m³")
print(f"  R²   : {metricas_lstm['R2']:.4f}")

In [ ]:
# ── Visualización continua del pronóstico multivariado ────────────────────────
# pred_full: tomamos el PRIMER paso de cada ventana para construir serie continua
pred_full   = y_pred_mv_inv[:, 0]      # primer paso de cada predicción
real_full   = y_te_mv_inv[:, 0]        # primer valor real de cada ventana

# Índice temporal alineado
index_cont = aq_test[FEATURES[0]].iloc[LAGS_MV : LAGS_MV + len(pred_full)].index

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=index_cont, y=real_full,
    name='Real', line=dict(color='black', width=1.5)
))
fig.add_trace(go.Scatter(
    x=index_cont, y=pred_full,
    name='Predicho (LSTM multivariado)', line=dict(color='crimson', width=1.5, dash='dash')
))
fig.update_layout(
    title=f'Pronóstico continuo — LSTM multivariado ({len(FEATURES)} features)',
    xaxis_title='Fecha', yaxis_title='O₃ (µg/m³)',
    template='plotly_white', height=450,
    xaxis=dict(rangeslider=dict(visible=True), type='date'),
    legend=dict(x=0.01, y=0.99)
)
fig.show()

## 12. Comparativa de Arquitecturas

### ¿Por qué comparar arquitecturas?

No existe una red neuronal universalmente superior para series de tiempo.
La elección depende del tamaño del dataset, la longitud de las dependencias,
la disponibilidad de variables exógenas y los recursos computacionales.

En esta sección entrenamos y comparamos cuatro arquitecturas fundamentales
usando exactamente el mismo conjunto de datos, parámetros de entrenamiento
y métricas de evaluación para garantizar una comparación justa.

| Arquitectura | Idea clave | Tipo de memoria |
|---|---|---|
| **MLP** | Aprende de rezagos aplanados, sin noción de secuencia | Ninguna (todo simultáneo) |
| **LSTM** | Celda de memoria con tres puertas, captura largo plazo | Explícita (cell state) |
| **GRU** | Versión simplificada de LSTM con dos puertas | Explícita (estado oculto) |
| **CNN-1D** | Detecta patrones locales con filtros convolucionales | Local (ventana del filtro) |

In [ ]:
# ── Definición de arquitecturas ──────────────────────────────────────────────
def build_mlp(lags):
    """
    MLP (Multilayer Perceptron): red densa que aplana los rezagos.
    No tiene noción de secuencia — cada rezago es un feature independiente.
    Es el baseline de red neuronal más simple.
    """
    return Sequential([
        Input(shape=(lags, 1)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(64,  activation='relu'),
        Dense(32,  activation='relu'),
        Dense(1)
    ], name='MLP')


def build_lstm(lags):
    """
    LSTM: procesa la secuencia paso a paso, manteniendo estado interno.
    Ideal para dependencias temporales largas.
    """
    return Sequential([
        Input(shape=(lags, 1)),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ], name='LSTM')


def build_gru(lags):
    """
    GRU: similar a LSTM pero con 2 puertas en lugar de 3.
    Más rápido, comparable en precisión para datasets de tamaño medio.
    """
    return Sequential([
        Input(shape=(lags, 1)),
        GRU(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ], name='GRU')


def build_cnn1d(lags):
    """
    CNN-1D: aplica filtros convolucionales a lo largo del tiempo.
    Detecta patrones locales repetidos (muy eficiente).
    padding='causal' garantiza que no se use información del futuro.
    """
    return Sequential([
        Input(shape=(lags, 1)),
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='causal'),
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='causal'),
        Conv1D(filters=32, kernel_size=3, activation='relu', padding='causal'),
        GlobalAveragePooling1D(),
        Dense(32, activation='relu'),
        Dense(1)
    ], name='CNN_1D')


print("Arquitecturas definidas: MLP, LSTM, GRU, CNN-1D")

In [ ]:
# ── Entrenamiento comparativo ─────────────────────────────────────────────────
# Mismo LAGS, mismos datos, misma función de pérdida, mismo EarlyStopping
# para garantizar comparación justa.

LAGS_COMP = LAGS   # 48 horas (definido en la sección 8)

X_tr_c, y_tr_c = crear_secuencias(train_sc, LAGS_COMP, 1)
X_vl_c, y_vl_c = crear_secuencias(val_sc,   LAGS_COMP, 1)
X_te_c, y_te_c = crear_secuencias(test_sc,  LAGS_COMP, 1)

builders = {
    'MLP':    build_mlp,
    'LSTM':   build_lstm,
    'GRU':    build_gru,
    'CNN-1D': build_cnn1d,
}

tabla_comp = {}
histories_comp = {}

for nombre, builder in builders.items():
    print(f"Entrenando {nombre}...", end=' ', flush=True)
    m = builder(LAGS_COMP)
    m.compile(optimizer=Adam(0.001), loss='mse')
    h = m.fit(
        X_tr_c, y_tr_c,
        validation_data=(X_vl_c, y_vl_c),
        epochs=100, batch_size=64,
        callbacks=[EarlyStopping(monitor='val_loss', patience=15,
                                 restore_best_weights=True)],
        verbose=0
    )
    y_hat_sc = m.predict(X_te_c, verbose=0)
    y_hat    = scaler_uni.inverse_transform(y_hat_sc)
    y_real_c = scaler_uni.inverse_transform(y_te_c)

    rmse_c  = np.sqrt(mean_squared_error(y_real_c, y_hat))
    mae_c   = mean_absolute_error(y_real_c, y_hat)
    r2_c    = r2_score(y_real_c, y_hat)
    smape_c = float(np.mean(2 * np.abs(y_hat - y_real_c) /
                            (np.abs(y_real_c) + np.abs(y_hat) + 1e-8)) * 100)
    n_params = m.count_params()

    tabla_comp[nombre] = {
        'RMSE (µg/m³)': round(rmse_c, 3),
        'MAE (µg/m³)' : round(mae_c, 3),
        'R²'          : round(r2_c, 4),
        'SMAPE (%)'   : round(smape_c, 2),
        'Parámetros'  : n_params,
        'Épocas'      : len(h.history['loss'])
    }
    histories_comp[nombre] = h
    print(f"RMSE={rmse_c:.3f}  R²={r2_c:.4f}  params={n_params:,}  epocas={len(h.history['loss'])}")

print("\nEntrenamiento comparativo completo.")

In [ ]:
# ── Tabla comparativa de resultados ──────────────────────────────────────────
df_comp = pd.DataFrame(tabla_comp).T
df_comp.index.name = 'Arquitectura'
df_comp_sorted = df_comp.sort_values('RMSE (µg/m³)')

print("\n" + "="*70)
print("  TABLA COMPARATIVA — Pronóstico O₃ 1 hora (datos de calidad del aire)")
print("="*70)
print(df_comp_sorted.to_string())
print("="*70)
mejor = df_comp_sorted.index[0]
print(f"\n  Mejor arquitectura (menor RMSE): {mejor}")

In [ ]:
# ── Gráficos comparativos ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colores_arq = {'MLP': '#4c72b0', 'LSTM': '#dd8452',
               'GRU': '#55a868', 'CNN-1D': '#c44e52'}

# Panel 1: RMSE
nombres = df_comp_sorted.index.tolist()
rmses   = df_comp_sorted['RMSE (µg/m³)'].tolist()
bars0 = axes[0,0].bar(nombres, rmses,
                       color=[colores_arq[n] for n in nombres],
                       edgecolor='black', alpha=0.85)
axes[0,0].bar_label(bars0, fmt='%.2f', fontsize=10, padding=3)
axes[0,0].set_title('RMSE (µg/m³) — menor es mejor', fontsize=12)
axes[0,0].set_ylabel('RMSE (µg/m³)')
axes[0,0].grid(axis='y', alpha=0.3)

# Panel 2: R²
r2s  = df_comp_sorted['R²'].tolist()
bars1 = axes[0,1].bar(nombres, r2s,
                       color=[colores_arq[n] for n in nombres],
                       edgecolor='black', alpha=0.85)
axes[0,1].bar_label(bars1, fmt='%.4f', fontsize=10, padding=3)
axes[0,1].set_title('R² — mayor es mejor', fontsize=12)
axes[0,1].set_ylabel('R²')
axes[0,1].set_ylim(max(0, min(r2s) - 0.05), 1.0)
axes[0,1].grid(axis='y', alpha=0.3)

# Panel 3: Curvas de pérdida comparativas
for nombre, h in histories_comp.items():
    axes[1,0].plot(h.history['val_loss'], label=nombre,
                   color=colores_arq[nombre], linewidth=1.5)
axes[1,0].set_title('Val-loss durante entrenamiento', fontsize=12)
axes[1,0].set_xlabel('Época')
axes[1,0].set_ylabel('MSE (val)')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Panel 4: Parámetros vs RMSE (eficiencia)
params = df_comp_sorted['Parámetros'].tolist()
for nombre, p, r in zip(nombres, params, rmses):
    axes[1,1].scatter(p, r, s=150, color=colores_arq[nombre],
                      edgecolors='black', zorder=5)
    axes[1,1].annotate(nombre, (p, r), textcoords='offset points',
                        xytext=(8, 4), fontsize=10)
axes[1,1].set_title('Parámetros vs RMSE (eficiencia)', fontsize=12)
axes[1,1].set_xlabel('Número de parámetros')
axes[1,1].set_ylabel('RMSE (µg/m³)')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Comparativa de arquitecturas — Pronóstico de O₃ 1 hora', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Predicciones visuales de todos los modelos ───────────────────────────────
# Re-predecimos para visualizar (mismos modelos ya entrenados)
n_mostrar = 24 * 7   # una semana

fig = go.Figure()
y_real_c_inv = scaler_uni.inverse_transform(y_te_c)
idx_vis = test_serie.index[LAGS_COMP : LAGS_COMP + len(y_real_c_inv)]

fig.add_trace(go.Scatter(
    x=idx_vis[-n_mostrar:], y=y_real_c_inv[-n_mostrar:, 0],
    name='Real', line=dict(color='black', width=2)
))

for nombre, builder in builders.items():
    # Volvemos a cargar el modelo ya entrenado via re-predict
    pass  # los modelos están en memoria porque se entrenaron arriba

# Nota: para graficar cada arquitectura, se requiere guardar referencias
# En el entrenamiento anterior se llama 'm' (último modelo = CNN-1D)
# A continuación una forma limpia de hacerlo:

resultados_pred = {}
for nombre, builder in builders.items():
    m2 = builder(LAGS_COMP)
    m2.compile(optimizer=Adam(0.001), loss='mse')
    # Entrenamiento rápido solo para visualización
    m2.fit(X_tr_c, y_tr_c, validation_data=(X_vl_c, y_vl_c),
           epochs=80, batch_size=64,
           callbacks=[EarlyStopping('val_loss', patience=12,
                                    restore_best_weights=True)],
           verbose=0)
    yh = scaler_uni.inverse_transform(m2.predict(X_te_c, verbose=0))
    resultados_pred[nombre] = yh

colores_px = {'MLP': 'blue', 'LSTM': 'red', 'GRU': 'green', 'CNN-1D': 'purple'}
for nombre, yh in resultados_pred.items():
    fig.add_trace(go.Scatter(
        x=idx_vis[-n_mostrar:], y=yh[-n_mostrar:, 0],
        name=nombre, line=dict(color=colores_px[nombre], width=1.2, dash='dash')
    ))

fig.update_layout(
    title='Predicciones de todas las arquitecturas — última semana del test',
    xaxis_title='Fecha', yaxis_title='O₃ (µg/m³)',
    template='plotly_white', height=450,
    xaxis=dict(rangeslider=dict(visible=True), type='date'),
    legend=dict(x=0.01, y=0.99)
)
fig.show()

## 13. Tabla comparativa del estado del arte

Más allá de las cuatro arquitecturas comparadas experimentalmente, existe un
ecosistema amplio de modelos de deep learning para series de tiempo. La siguiente
tabla resume las opciones más relevantes del estado del arte.

| Arquitectura | Tipo | Memoria temporal | Multi-step | Variables exógenas | Implementación | Mejor para |
|---|---|---|---|---|---|---|
| **MLP + rezagos** | Feedforward | No (todo simultáneo) | Directo | Sí | `keras.Dense` | Baseline rápido, relaciones no lineales simples |
| **RNN simple** | Recurrente | Corta (gradiente desvaneciente) | Sí | Sí | `keras.SimpleRNN` | Raramente; superada por LSTM/GRU |
| **LSTM** | Recurrente | Larga (cell state) | Sí | Sí | `keras.LSTM` | Series largas, dependencias complejas |
| **GRU** | Recurrente simplificada | Larga (2 puertas) | Sí | Sí | `keras.GRU` | Como LSTM, más rápido con datos medianos |
| **CNN-1D** | Convolucional | Local (tamaño del kernel) | Sí | Sí | `keras.Conv1D` | Patrones locales repetitivos, muy eficiente |
| **TCN** | Convolución dilatada causal | Larga (dilations) | Sí | Sí | `pip install keras-tcn` | Alternativa eficiente a LSTM, paralelizable |
| **Transformer** | Auto-atención | Global (todos vs todos) | Sí | Sí | `keras.MultiHeadAttention` | Dependencias de largo alcance, datos abundantes |
| **N-BEATS** | Feedforward con bloques residuales | Implícita (stacks) | Sí | No (vanilla) | `pip install neuralforecast` | Pronóstico univariado de alta precisión |
| **N-HiTS** | Bloques jerárquicos | Multi-escala | Sí | No (vanilla) | `pip install neuralforecast` | Horizontes largos, jerarquía temporal |
| **DeepAR** | RNN + distribución probabilística | Explícita | Sí | Sí | `pip install gluonts` | Intervalos de predicción, múltiples series |
| **TFT** | Transformer + atención temporal | Global + local | Sí | Sí (clave) | `pip install pytorch-forecasting` | Múltiples series con covariables conocidas |
| **PatchTST** | Transformer sobre parches | Global (parches) | Sí | Parcial | `pip install neuralforecast` | Datasets grandes, preentrenamiento |
| **TimesNet** | Transformación 2D temporal | Multi-escala 2D | Sí | Sí | `pip install neuralforecast` | Captura de periodicidades 2D |

### Guía de selección

```
¿Tienes muchas variables exógenas con dinámicas conocidas?
  → Temporal Fusion Transformer (TFT)

¿Necesitas intervalos de predicción probabilísticos?
  → DeepAR o Conformalized Forecasting

¿Horizonte largo (> 48 h) con un dataset grande?
  → N-HiTS, PatchTST o Transformer

¿Dataset mediano (< 50K puntos) y velocidad es prioritaria?
  → GRU o CNN-1D

¿Quieres alta precisión en 1 serie univariada?
  → N-BEATS

¿Estás enseñando o prototipando?
  → LSTM es el punto de partida estándar
```

## 14. Conclusiones e Interpretación

### Resultados obtenidos

1. **LSTM univariado 1-paso:** Establece la línea base del modelo neuronal.
   Si RMSE_LSTM < RMSE_persistencia, el modelo agrega valor sobre el baseline más
   simple. Típicamente los modelos neuronales mejoran entre 20-40% sobre la
   persistencia en datos de calidad del aire.

2. **Multi-step:** El RMSE aumenta progresivamente con el horizonte H.
   Esto es esperable y no es un defecto del modelo — refleja la incertidumbre
   inherente del futuro. Un modelo que no degrada con H probablemente está
   haciendo predicciones constantes (media global).

3. **Multivariado (N:1):** La inclusión de NO₂, CO, PM2.5 y las componentes del
   viento puede mejorar el pronóstico si estas variables preceden causalmente al O₃.
   Si no hay mejora, es posible que la relación sea simultánea (misma hora) y no
   predictiva.

4. **Comparativa MLP vs LSTM vs GRU vs CNN-1D:**
   - En series con ciclos diurnos fuertes (como O₃), la CNN-1D a menudo compite
     de igual a igual con LSTM por su capacidad de detectar patrones locales.
   - GRU suele igualar a LSTM con menos parámetros y tiempo de entrenamiento.
   - MLP puede sorprender con buen rendimiento si los lags contienen suficiente
     información (el problema es esencialmente lineal en los rezagos).

### Limitaciones importantes

- **Sin validación cruzada temporal (walk-forward):** Un único test puede ser
  optimista o pesimista según la estación. El estándar de oro es evaluar sobre
  múltiples ventanas temporales.
- **Sin intervalos de predicción:** Los modelos presentados son deterministas.
  En aplicaciones reales (alertas de contaminación), es crítico cuantificar la
  incertidumbre.
- **Hiperparámetros sin búsqueda formal:** El número de capas, unidades, dropout
  y learning rate se eligieron manualmente. Un RandomSearch o Optuna mejoraría
  los resultados.

### Próximos pasos

```
1. Implementar walk-forward cross-validation
2. Explorar TCN y Transformer con atención
3. Agregar variables meteorológicas externas (temperatura, radiación solar)
4. Implementar predicción probabilística con DeepAR o MC-Dropout
5. Comparar con modelos clásicos: SARIMA, Prophet, XGBoost con rezagos
```

---

### Recursos adicionales

- [Skforecast — Deep Learning para series de tiempo](https://cienciadedatos.net/documentos/py35-redes-neuronales-python)
- [NeuralForecast — N-BEATS, N-HiTS, TFT](https://nixtlaverse.nixtla.io/neuralforecast/index.html)
- [Pytorch Forecasting — TFT](https://pytorch-forecasting.readthedocs.io/)
- [GluonTS — DeepAR](https://ts.gluon.ai/)